# AQRA — Known-Factor Reproduction Notebook

This notebook reproduces well-documented equity factors using the AQRA pipeline.

## Goals
1. Load S&P 500 prices and fundamentals.
2. Build Lane S (structural alpha) features.
3. Run backtests for momentum, value, and quality signals.
4. Compare information coefficient (IC) and Sharpe ratios to literature.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from aqra.config import load_config
from aqra.db import AQRADatabase
from aqra.data.cache import DataCache
from aqra.features.lane_s import LaneSFeatureBuilder
from aqra.signals.lane_s_signals import LaneSSignalLibrary
from aqra.backtest.lane_s_bt import LaneSBacktest
from aqra.utils import annualized_sharpe

cfg = load_config()
db = AQRADatabase(f"{cfg.data_dir}/aqra.duckdb")
cache = DataCache(db, cfg)


## 1. Load Universe & Prices

Use `DataCache.refresh_prices` to populate `raw_prices`. For a quick reproduction, limit to a small universe or a single ticker.

In [ ]:
# Example: fetch AAPL prices for a multi-year window
cache.refresh_prices(start="2020-01-01", end="2024-12-31", tickers=["AAPL"])
prices = db.conn.execute("SELECT * FROM raw_prices ORDER BY date").fetchdf()
prices.tail()

## 2. Build Lane S Features

Momentum (12-1 month), value placeholders, quality placeholders, low-vol, insider, and macro regime.

In [ ]:
builder = LaneSFeatureBuilder(db)
features = builder.build("2020-06-01", "2024-12-31")
features[["ticker", "date", "mom_12_1", "pe_rank", "pb_rank", "quality_score"]].dropna().head()

## 3. Run Signal Backtests

Iterate through Lane S signal candidates and report IC, Sharpe, and max drawdown.

In [ ]:
lib = LaneSSignalLibrary()
candidates = lib.generate()
runner = LaneSBacktest(db)

results = []
for cand in candidates:
    result = runner.run(cand, start="2020-06-01", end="2024-12-31", holding_period=21, cost_bps=10)
    if not result:
        continue
    results.append({
        "id": cand.id,
        "name": cand.name,
        "sharpe": result.get("sharpe"),
        "ic": result.get("ic"),
        "max_drawdown": result.get("max_drawdown"),
    })

pd.DataFrame(results)

## 4. Literature Sanity Checks

- **Momentum (12-1):** positive mean IC, positive Sharpe in most windows.
- **Value:** positive mean IC (higher P/E/B ranks predict lower returns, i.e., cheaper stocks outperform).
- **Quality:** positive mean IC, defensive Sharpe.

If any factor shows the wrong sign across a long sample, investigate look-ahead bias or data alignment.

In [ ]:
# Placeholder: plot IC by signal
import matplotlib.pyplot as plt
if results:
    df = pd.DataFrame(results)
    df.set_index("id")[["ic"]].plot(kind="bar", title="Mean IC by Signal")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.show()